# 🏦 Банковская система: `Account` и `CreditAccount`

## О проекте

Проект моделирует работу обычного и кредитного банковских счетов.

Программа поддерживает:

- создание обычного банковского счёта;
- создание счёта с кредитным лимитом;
- пополнение и снятие денежных средств;
- точные денежные расчёты с использованием `Decimal`;
- проверку доступного остатка и кредитного лимита;
- регистрацию успешных и неуспешных транзакций;
- блокировку и разблокировку счёта;
- проверку полномочий владельца, сотрудника банка и системы;
- хранение истории финансовых операций;
- хранение истории событий счёта;
- неизменяемые записи транзакций и событий;
- время операций в UTC;
- вывод времени в часовом поясе Москвы;
- форматированный текстовый журнал;
- автоматические тесты;
- демонстрационный сценарий работы программы.

---

# 1. Цели проекта

Основная цель проекта — показать применение объектно-ориентированного
программирования на примере банковской системы.

В работе используются:

- классы и наследование;
- инкапсуляция;
- приватные атрибуты;
- переопределение методов;
- полиморфизм;
- композиция объектов;
- `dataclass`;
- неизменяемые объекты;
- перечисления `StrEnum`;
- аннотации типов;
- исключения;
- точные вычисления через `Decimal`;
- UUID-идентификаторы;
- даты и часовые пояса;
- история изменений состояния объекта;
- автоматическое тестирование.

---

# 2. Денежные значения

Для хранения денежных значений используется тип `Decimal`.

```python
from decimal import Decimal
```

Пример создания суммы:

```python
amount = Decimal("1000.00")
```

Использование строки важно, потому что создание `Decimal` из `float`
может перенести в значение погрешность двоичного представления числа.

## Денежные константы

```python
ZERO_MONEY = Decimal("0.00")
MONEY_QUANTUM = Decimal("0.01")
```

- `ZERO_MONEY` представляет нулевую денежную сумму;
- `MONEY_QUANTUM` определяет точность до двух знаков после запятой.

## Нормализация

Функция `normalize_money()`:

- принимает только `Decimal`;
- отклоняет бесконечность и `NaN`;
- округляет значение до двух знаков;
- использует банковское округление `ROUND_HALF_EVEN`.

```python
normalize_money(Decimal("10.125"))
```

Результат:

```text
Decimal("10.12")
```

---

# 3. Перечисления

Вместо строковых литералов используются перечисления `StrEnum`.

Это снижает вероятность опечаток и упрощает статическую проверку типов.

## `OperationType`

Тип банковской операции:

- `DEPOSIT` — пополнение;
- `WITHDRAW` — снятие.

## `OperationStatus`

Результат банковской операции:

- `SUCCESS` — операция выполнена;
- `FAIL` — в выполнении операции отказано.

## `FailureReason`

Причина отказа:

- `INSUFFICIENT_FUNDS` — недостаточно собственных средств;
- `CREDIT_LIMIT_EXCEEDED` — превышен кредитный лимит;
- `ACCOUNT_BLOCKED` — счёт заблокирован.

## `Currency`

Валюта банковского счёта:

- `RUB`;
- `USD`;
- `EUR`.

В рамках одного счёта все суммы хранятся в одной валюте. Конвертация
валют в проекте не выполняется.

## `AccountStatus`

Состояние банковского счёта:

- `ACTIVE` — счёт активен;
- `BLOCKED` — счёт заблокирован.

## `ActorType`

Тип инициатора события:

- `ACCOUNT_HOLDER` — владелец счёта;
- `BANK_EMPLOYEE` — сотрудник банка;
- `SYSTEM` — банковская система.

## `AccountEventType`

Тип события счёта:

- `BLOCKED` — счёт заблокирован;
- `UNBLOCKED` — счёт разблокирован.

## `BlockReason`

Причина блокировки:

- `USER_REQUEST` — запрос владельца;
- `FRAUD_SUSPICION` — подозрение на мошенничество;
- `BANK_DECISION` — решение банка;
- `LEGAL_RESTRICTION` — юридическое ограничение.

---

# 4. Инициатор события

Инициатор блокировки или разблокировки представлен объектом
`EventActor`.

```python
@dataclass(frozen=True, slots=True)
class EventActor:
    actor_type: ActorType
    actor_id: UUID | None = None
```

Объект является неизменяемым.

## Правила идентификации

Для системы идентификатор не указывается:

```python
system = EventActor(
    actor_type=ActorType.SYSTEM,
)
```

Владелец и сотрудник банка должны иметь UUID:

```python
employee = EventActor(
    actor_type=ActorType.BANK_EMPLOYEE,
    actor_id=uuid4(),
)
```

Следующие состояния запрещены:

- система с `actor_id`;
- владелец без `actor_id`;
- сотрудник без `actor_id`.

Согласованность данных проверяется в `__post_init__()`.

---

# 5. Модель транзакции

Каждая финансовая операция представлена объектом `Transaction`.

Транзакция содержит:

| Поле | Тип | Назначение |
|---|---|---|
| `transaction_id` | `UUID` | Уникальный идентификатор транзакции |
| `account_id` | `UUID` | Идентификатор счёта |
| `currency` | `Currency` | Валюта операции |
| `operation_type` | `OperationType` | Тип операции |
| `amount` | `Decimal` | Сумма операции |
| `created_at` | `datetime` | Время создания |
| `balance_after` | `Decimal` | Баланс после операции |
| `status` | `OperationStatus` | Результат операции |
| `failure_reason` | `FailureReason \| None` | Причина отказа |
| `credit_used` | `bool \| None` | Использование кредитных средств |

`Transaction` объявлен с параметрами:

```python
@dataclass(frozen=True, slots=True)
```

Это означает:

- поля транзакции нельзя изменить после создания;
- невозможно задним числом изменить сумму или статус;
- `slots=True` ограничивает набор допустимых атрибутов.

## Согласованность статуса

Для транзакции действуют правила:

- успешная операция не может содержать причину отказа;
- неуспешная операция обязана содержать причину отказа.

Свойство `is_successful` позволяет проверить результат:

```python
if transaction.is_successful:
    print("Операция выполнена")
```

---

# 6. События банковского счёта

Блокировка и разблокировка не являются денежными транзакциями.
Для них используется отдельная модель `AccountEvent`.

Событие содержит:

| Поле | Тип | Назначение |
|---|---|---|
| `event_id` | `UUID` | Уникальный идентификатор события |
| `account_id` | `UUID` | Идентификатор счёта |
| `event_type` | `AccountEventType` | Тип события |
| `created_at` | `datetime` | Время события |
| `actor` | `EventActor` | Инициатор события |
| `block_reason` | `BlockReason \| None` | Причина блокировки |

Для событий действуют правила:

- событие блокировки обязано содержать причину;
- событие разблокировки не может содержать причину блокировки.

`AccountEvent` является неизменяемым объектом.

---

# 7. Класс `Account`

Класс `Account` моделирует обычный банковский счёт.

## Создание счёта

```python
account = Account(
    account_holder="Илья",
    balance=Decimal("1000.00"),
    currency=Currency.RUB,
)
```

Полная сигнатура конструктора:

```python
Account(
    account_holder,
    balance=ZERO_MONEY,
    currency=Currency.RUB,
    account_id=None,
    account_status=AccountStatus.ACTIVE,
    block_reason=None,
    holder_id=None,
)
```

## Основные данные счёта

Счёт хранит:

- имя владельца;
- UUID владельца;
- UUID счёта;
- валюту;
- текущий баланс;
- текущий статус;
- причину блокировки;
- историю транзакций;
- историю событий.

Если `account_id` или `holder_id` не переданы, программа создаёт UUID
автоматически.

## Проверка начального состояния

При создании счёта проверяется:

- баланс должен иметь тип `Decimal`;
- начальный баланс не может быть отрицательным;
- идентификаторы должны иметь тип `UUID`;
- валюта должна иметь тип `Currency`;
- статус должен иметь тип `AccountStatus`;
- причина блокировки должна иметь тип `BlockReason`;
- заблокированный счёт обязан иметь причину блокировки;
- активный счёт не может иметь причину блокировки.

---

# 8. Пополнение счёта

Метод:

```python
account.deposit(amount)
```

Правила:

- `amount` должен иметь тип `Decimal`;
- сумма должна быть больше нуля;
- сумма нормализуется до двух знаков;
- баланс увеличивается;
- создаётся успешная транзакция;
- транзакция добавляется в историю;
- метод возвращает созданный объект `Transaction`.

Пример:

```python
transaction = account.deposit(
    Decimal("250.50")
)
```

---

# 9. Снятие со счёта

Метод:

```python
account.withdraw(amount)
```

Правила:

- сумма должна иметь тип `Decimal`;
- сумма должна быть больше нуля;
- сумма нормализуется до двух знаков;
- со счёта нельзя снять больше доступного остатка;
- с заблокированного счёта нельзя снять деньги;
- успешная операция уменьшает баланс;
- неуспешная попытка не изменяет баланс;
- любая попытка снятия фиксируется в истории;
- метод возвращает объект `Transaction`.

## Недостаточно денег

Если средств недостаточно:

- метод не выбрасывает исключение;
- баланс остаётся прежним;
- создаётся транзакция со статусом `FAIL`;
- указывается причина `INSUFFICIENT_FUNDS`.

Такой результат является ожидаемым результатом банковской операции,
а не программной ошибкой.

## Заблокированный счёт

При попытке снятия с заблокированного счёта:

- баланс не меняется;
- операция получает статус `FAIL`;
- указывается причина `ACCOUNT_BLOCKED`.

---

# 10. Программные ошибки и банковские отказы

В программе разделяются два вида ошибок.

## Программная ошибка

Если вызывающий код передал неправильные данные, выбрасывается
исключение:

- `TypeError` — неправильный тип аргумента;
- `ValueError` — недопустимое значение;
- `PermissionError` — у инициатора нет полномочий.

Примеры:

- вместо `Decimal` передан `float`;
- передана отрицательная сумма;
- вместо UUID передана строка;
- клиент пытается управлять чужим счётом.

## Банковский отказ

Если запрос корректен, но банк не может выполнить операцию, создаётся
неуспешная транзакция:

- недостаточно денег;
- превышен кредитный лимит;
- счёт заблокирован.

Это позволяет сохранить попытку в истории.

---

# 11. Блокировка счёта

Метод:

```python
account.block_account(
    block_reason,
    actor,
)
```

Метод:

- проверяет тип причины;
- проверяет тип инициатора;
- проверяет полномочия инициатора;
- изменяет статус на `BLOCKED`;
- сохраняет причину;
- создаёт `AccountEvent`;
- возвращает созданное событие.

Если счёт уже заблокирован, повторное изменение состояния не
выполняется.

## Полномочия на блокировку

| Причина | Владелец | Сотрудник | Система |
|---|:---:|:---:|:---:|
| Запрос владельца | ✅ | ✅ | ❌ |
| Подозрение на мошенничество | ❌ | ✅ | ✅ |
| Решение банка | ❌ | ✅ | ❌ |
| Юридическое ограничение | ❌ | ✅ | ✅ |

Владелец может управлять только своим счётом. Его `actor_id` должен
совпадать с `holder_id` счёта.

---

# 12. Разблокировка счёта

Метод:

```python
account.unblock_account(actor)
```

Правила:

- блокировку по запросу владельца может снять владелец этого счёта
  или сотрудник банка;
- блокировку по другим причинам может снять только сотрудник банка;
- система не может самостоятельно отменить созданную блокировку;
- после разблокировки статус меняется на `ACTIVE`;
- текущая причина блокировки очищается;
- в историю добавляется событие `UNBLOCKED`.

Если счёт уже активен, повторное изменение состояния не выполняется.

---

# 13. История операций и событий

## История транзакций

```python
history = account.get_history()
```

Метод возвращает:

```python
tuple[Transaction, ...]
```

Возвращается кортеж, поэтому вызывающий код не может добавить или
удалить записи из внутренней истории.

## История событий

```python
events = account.get_events_history()
```

Метод возвращает:

```python
tuple[AccountEvent, ...]
```

Финансовые транзакции и административные события хранятся отдельно.

---

# 14. Класс `CreditAccount`

`CreditAccount` наследуется от `Account`:

```python
class CreditAccount(Account):
    ...
```

Кредитный счёт поддерживает всю функциональность обычного счёта и
добавляет кредитный лимит.

## Создание кредитного счёта

```python
credit_account = CreditAccount(
    account_holder="Илья",
    balance=Decimal("1000.00"),
    credit_limit=Decimal("500.00"),
    currency=Currency.RUB,
)
```

## Доступные средства

Доступная сумма рассчитывается по формуле:

```text
баланс + кредитный лимит
```

Метод:

```python
credit_account.get_available_funds()
```

Пример:

```text
Баланс: 1000.00 RUB
Кредитный лимит: 500.00 RUB
Доступно: 1500.00 RUB
```

## Превышение лимита

Если запрошенная сумма превышает доступные средства:

- баланс не изменяется;
- создаётся неуспешная транзакция;
- статус равен `FAIL`;
- причина равна `CREDIT_LIMIT_EXCEEDED`.

## Использование кредита

Поле `credit_used` показывает, привело ли успешное снятие к
отрицательному балансу.

---

# 15. Дата, время и часовые пояса

Время транзакций и событий создаётся в UTC:

```python
datetime.now(timezone.utc)
```

Хранение времени в UTC исключает неоднозначность и позволяет
преобразовывать дату для разных регионов.

Для отображения используется московский часовой пояс:

```python
ZoneInfo("Europe/Moscow")
```

Функция `format_moscow_datetime()` формирует строку вида:

```text
10.10.2026 15:42:17 MSK (UTC+03:00, Europe/Moscow)
```

Исходное UTC-время в объектах при этом не изменяется.

---

# 16. Форматирование журнала

Для отображения данных используются отдельные функции.

## Денежное значение

```python
format_money(value, currency)
```

Пример:

```text
1 250.50 RUB
```

## История транзакций

```python
print(format_transaction_history(account))
```

Выводит:

- UUID транзакции;
- тип операции;
- сумму;
- статус;
- баланс после операции;
- причину отказа;
- использование кредитных средств;
- московское время.

## История событий

```python
print(format_account_events(account))
```

Выводит:

- UUID события;
- тип события;
- инициатора;
- UUID инициатора;
- причину блокировки;
- московское время.

## Полный журнал

```python
print(format_account_journal(account))
```

Функция объединяет историю финансовых транзакций и историю событий
счёта.

Функции форматирования находятся вне классов, потому что отвечают за
представление данных, а не за изменение состояния банковского счёта.

---

# 17. Строковое представление

Классы `Account` и `CreditAccount` реализуют метод `__str__()`.

Пример:

```python
print(account)
```

Обычный счёт выводит:

- владельца;
- текущий баланс;
- валюту.

Кредитный счёт дополнительно выводит:

- кредитный лимит;
- общую сумму доступных средств.

---

# 18. Автоматические тесты

В проекте предусмотрены тесты для проверки:

- нормализации денежных значений;
- создания обычного счёта;
- валидации конструктора;
- успешного пополнения;
- запрета нулевого и отрицательного пополнения;
- успешного снятия;
- отказа при недостатке средств;
- работы кредитного лимита;
- проверки `EventActor`;
- полномочий владельца;
- полномочий сотрудника и системы;
- снятия с заблокированного счёта;
- защиты истории;
- неизменяемости транзакций.

Запуск:

```python
run_all_tests()
```

Ожидаемый результат:

```text
Все тесты пройдены: 13
```

---

# 19. Демонстрация

Для запуска полного демонстрационного сценария используется:

```python
demonstrate_banking_program()
```

Демонстрация последовательно показывает:

1. создание кредитного счёта;
2. пополнение;
3. обычное снятие;
4. использование кредитных средств;
5. отказ при превышении кредитного лимита;
6. погашение задолженности;
7. блокировку по запросу владельца;
8. отказ в снятии с заблокированного счёта;
9. разблокировку владельцем;
10. системную блокировку;
11. запрет системе самостоятельно снять блокировку;
12. разблокировку сотрудником банка;
13. итоговое состояние счёта;
14. полный журнал транзакций и событий.

---

# 20. Пример использования

```python
holder_id = uuid4()

owner = EventActor(
    actor_type=ActorType.ACCOUNT_HOLDER,
    actor_id=holder_id,
)

account = CreditAccount(
    account_holder="Илья",
    balance=Decimal("1000.00"),
    credit_limit=Decimal("500.00"),
    currency=Currency.RUB,
    holder_id=holder_id,
)

account.deposit(Decimal("250.50"))
account.withdraw(Decimal("400.00"))

account.block_account(
    block_reason=BlockReason.USER_REQUEST,
    actor=owner,
)

print(format_account_journal(account))
```

---

# 21. Проверка качества кода

Для статической проверки Notebook используется Ruff:

```bash
ruff check z5.ipynb
```

Для проверки форматирования:

```bash
ruff format --check z5.ipynb
```

Для автоматического форматирования:

```bash
ruff format z5.ipynb
```

Основные требования:

- отступ — четыре пробела;
- понятные имена классов и методов;
- аннотации типов;
- строки документации;
- отсутствие неиспользуемых импортов;
- ограниченная длина строк;
- отсутствие лишних пробелов;
- сравнение перечислений через `is`;
- единый стиль форматирования.

---

# 22. Итог

В результате создана расширенная объектно-ориентированная модель
банковского счёта.

Проект выходит за рамки первоначального задания, потому что включает:

- точную денежную арифметику;
- типизированные статусы;
- неизменяемые модели;
- уникальные идентификаторы;
- аудит финансовых операций;
- аудит административных событий;
- блокировку и разблокировку;
- разграничение полномочий;
- проверку конкретного владельца;
- поддержку часовых поясов;
- форматированный журнал;
- автоматические тесты;
- демонстрационный сценарий.

In [18]:
from dataclasses import dataclass
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_EVEN
from enum import StrEnum
from uuid import UUID, uuid4


ZERO_MONEY = Decimal("0.00")
MONEY_QUANTUM = Decimal("0.01")


def normalize_money(value: object) -> Decimal:
    """Проверяет и округляет денежное значение до двух знаков."""

    if not isinstance(value, Decimal):
        raise TypeError("Денежное значение должно иметь тип Decimal")

    if not value.is_finite():
        raise ValueError("Денежное значение должно быть конечным числом")

    return value.quantize(
        MONEY_QUANTUM,
        rounding=ROUND_HALF_EVEN,
    )


class Currency(StrEnum):
    """Определяет валюту банковского счёта."""

    RUB = "RUB"
    USD = "USD"
    EUR = "EUR"


class OperationType(StrEnum):
    """Определяет тип банковской операции."""

    DEPOSIT = "deposit"
    WITHDRAW = "withdraw"


class OperationStatus(StrEnum):
    """Определяет результат банковской операции."""

    SUCCESS = "success"
    FAIL = "fail"


class FailureReason(StrEnum):
    """Определяет причину отказа в операции."""

    INSUFFICIENT_FUNDS = "insufficient_funds"
    CREDIT_LIMIT_EXCEEDED = "credit_limit_exceeded"
    ACCOUNT_BLOCKED = "account_blocked"


class AccountStatus(StrEnum):
    """Определяет состояние банковского счёта."""

    ACTIVE = "active"
    BLOCKED = "blocked"


class BlockReason(StrEnum):
    """Определяет причину блокировки счёта."""

    USER_REQUEST = "user_request"
    FRAUD_SUSPICION = "fraud_suspicion"
    BANK_DECISION = "bank_decision"
    LEGAL_RESTRICTION = "legal_restriction"


class ActorType(StrEnum):
    """Определяет тип инициатора события."""

    ACCOUNT_HOLDER = "account_holder"
    BANK_EMPLOYEE = "bank_employee"
    SYSTEM = "system"


class AccountEventType(StrEnum):
    """Определяет тип события банковского счёта."""

    BLOCKED = "blocked"
    UNBLOCKED = "unblocked"


@dataclass(frozen=True, slots=True)
class EventActor:
    """Представляет инициатора события счёта."""

    actor_type: ActorType
    actor_id: UUID | None = None

    def __post_init__(self) -> None:
        """Проверяет согласованность типа и ID инициатора."""

        if not isinstance(self.actor_type, ActorType):
            raise TypeError("Тип инициатора должен иметь тип ActorType")

        if self.actor_type is ActorType.SYSTEM and self.actor_id is not None:
            raise ValueError("Системный инициатор не должен иметь actor_id")

        if self.actor_type is not ActorType.SYSTEM and self.actor_id is None:
            raise ValueError("Владелец или сотрудник должен иметь actor_id")

        if self.actor_id is not None and not isinstance(self.actor_id, UUID):
            raise TypeError("Идентификатор инициатора должен иметь тип UUID")


@dataclass(frozen=True, slots=True)
class Transaction:
    """Представляет неизменяемую банковскую операцию."""

    transaction_id: UUID
    account_id: UUID
    currency: Currency
    operation_type: OperationType
    amount: Decimal
    created_at: datetime
    balance_after: Decimal
    status: OperationStatus
    failure_reason: FailureReason | None = None
    credit_used: bool | None = None

    def __post_init__(self) -> None:
        """Проверяет корректность банковской транзакции."""

        if not isinstance(self.transaction_id, UUID):
            raise TypeError("Идентификатор транзакции должен иметь тип UUID")

        if not isinstance(self.account_id, UUID):
            raise TypeError("Идентификатор счёта должен иметь тип UUID")

        if not isinstance(self.currency, Currency):
            raise TypeError("Валюта транзакции должна иметь тип Currency")

        if not isinstance(
            self.operation_type,
            OperationType,
        ):
            raise TypeError("Тип операции должен иметь тип OperationType")

        if not isinstance(self.amount, Decimal):
            raise TypeError("Сумма транзакции должна иметь тип Decimal")

        if not isinstance(
            self.balance_after,
            Decimal,
        ):
            raise TypeError("Баланс после операции должен иметь тип Decimal")

        if not isinstance(self.status, OperationStatus):
            raise TypeError("Статус операции должен иметь тип OperationStatus")

        if self.failure_reason is not None and not isinstance(
            self.failure_reason,
            FailureReason,
        ):
            raise TypeError("Причина отказа должна иметь тип FailureReason")

        if self.credit_used is not None and not isinstance(self.credit_used, bool):
            raise TypeError("Признак использования кредита должен иметь тип bool")

        if not isinstance(self.created_at, datetime):
            raise TypeError("Время транзакции должно иметь тип datetime")

        if self.created_at.tzinfo is None or self.created_at.utcoffset() is None:
            raise ValueError("Время транзакции должно содержать часовой пояс")

        if not self.amount.is_finite():
            raise ValueError("Сумма транзакции должна быть конечным числом")

        if not self.balance_after.is_finite():
            raise ValueError("Баланс после операции должен быть конечным числом")

        if self.amount <= ZERO_MONEY:
            raise ValueError("Сумма транзакции должна быть положительной")

        normalized_amount = normalize_money(self.amount)
        normalized_balance = normalize_money(self.balance_after)

        if normalized_amount != self.amount:
            raise ValueError(
                "Сумма транзакции должна содержать не более двух знаков после запятой"
            )

        if normalized_balance != self.balance_after:
            raise ValueError(
                "Баланс после операции должен содержать "
                "не более двух знаков после запятой"
            )

        if self.status is OperationStatus.SUCCESS and self.failure_reason is not None:
            raise ValueError("Успешная операция не может иметь причину отказа")

        if self.status is OperationStatus.FAIL and self.failure_reason is None:
            raise ValueError("Неуспешная операция должна иметь причину отказа")

    @property
    def is_successful(self) -> bool:
        """Показывает, была ли операция успешной."""

        return self.status is OperationStatus.SUCCESS


@dataclass(frozen=True, slots=True)
class AccountEvent:
    """Представляет неизменяемое событие счёта."""

    event_id: UUID
    account_id: UUID
    event_type: AccountEventType
    created_at: datetime
    actor: EventActor
    block_reason: BlockReason | None = None

    def __post_init__(self) -> None:
        """Проверяет корректность события счёта."""

        if not isinstance(self.event_id, UUID):
            raise TypeError("Идентификатор события должен иметь тип UUID")

        if not isinstance(self.account_id, UUID):
            raise TypeError("Идентификатор счёта должен иметь тип UUID")

        if not isinstance(
            self.event_type,
            AccountEventType,
        ):
            raise TypeError("Тип события должен иметь тип AccountEventType")

        if not isinstance(self.created_at, datetime):
            raise TypeError("Время события должно иметь тип datetime")

        if self.created_at.tzinfo is None or self.created_at.utcoffset() is None:
            raise ValueError("Время события должно содержать часовой пояс")

        if not isinstance(self.actor, EventActor):
            raise TypeError("Инициатор должен иметь тип EventActor")

        if self.block_reason is not None and not isinstance(
            self.block_reason,
            BlockReason,
        ):
            raise TypeError("Причина блокировки должна иметь тип BlockReason")

        if self.event_type is AccountEventType.BLOCKED and self.block_reason is None:
            raise ValueError("Событие блокировки должно иметь причину")

        if (
            self.event_type is AccountEventType.UNBLOCKED
            and self.block_reason is not None
        ):
            raise ValueError(
                "Событие разблокировки не может содержать причину блокировки"
            )


class Account:
    """Моделирует банковский счёт и хранит историю операций."""

    def __init__(
        self,
        account_holder: str,
        balance: Decimal = ZERO_MONEY,
        *,
        currency: Currency = Currency.RUB,
        holder_id: UUID | None = None,
        account_id: UUID | None = None,
        account_status: AccountStatus = AccountStatus.ACTIVE,
        block_reason: BlockReason | None = None,
    ) -> None:
        """Создаёт банковский счёт."""

        if not isinstance(account_holder, str):
            raise TypeError("Имя владельца должно иметь тип str")

        account_holder = account_holder.strip()

        if not account_holder:
            raise ValueError("Имя владельца не может быть пустым")

        balance = normalize_money(balance)

        if balance < ZERO_MONEY:
            raise ValueError("Начальный баланс не может быть отрицательным")

        if not isinstance(currency, Currency):
            raise TypeError("Валюта должна иметь тип Currency")

        if holder_id is not None and not isinstance(holder_id, UUID):
            raise TypeError("Идентификатор владельца должен иметь тип UUID")

        if account_id is not None and not isinstance(account_id, UUID):
            raise TypeError("Идентификатор счёта должен иметь тип UUID")

        if not isinstance(account_status, AccountStatus):
            raise TypeError("Статус счёта должен иметь тип AccountStatus")

        if block_reason is not None and not isinstance(block_reason, BlockReason):
            raise TypeError("Причина блокировки должна иметь тип BlockReason")

        if account_status is AccountStatus.BLOCKED and block_reason is None:
            raise ValueError(
                "Для заблокированного счёта необходимо указать причину блокировки"
            )

        if account_status is AccountStatus.ACTIVE and block_reason is not None:
            raise ValueError("Активный счёт не может иметь причину блокировки")

        self.holder: str = account_holder

        self._holder_id: UUID = uuid4() if holder_id is None else holder_id
        self._account_id: UUID = uuid4() if account_id is None else account_id

        self._currency: Currency = currency
        self._balance: Decimal = balance
        self._account_status: AccountStatus = account_status
        self._block_reason: BlockReason | None = block_reason

        self._operations_history: list[Transaction] = []
        self._events_history: list[AccountEvent] = []

    def __str__(self) -> str:
        """Возвращает строковое представление счёта."""

        return (
            f"Владелец: {self.holder}. Баланс: {self._balance} {self._currency.value}"
        )

    # Получение основной информации

    def get_account_id(self) -> UUID:
        """Возвращает идентификатор счёта."""

        return self._account_id

    def get_holder_id(self) -> UUID:
        """Возвращает идентификатор владельца счёта."""

        return self._holder_id

    def get_currency(self) -> Currency:
        """Возвращает валюту счёта."""

        return self._currency

    def get_balance(self) -> Decimal:
        """Возвращает текущий баланс счёта."""

        return self._balance

    def get_account_status(self) -> AccountStatus:
        """Возвращает статус банковского счёта."""

        return self._account_status

    def get_block_reason(self) -> BlockReason | None:
        """Возвращает текущую причину блокировки."""

        return self._block_reason

    @property
    def is_blocked(self) -> bool:
        """Показывает, заблокирован ли счёт."""

        return self._account_status is AccountStatus.BLOCKED

    # Финансовые операции

    def deposit(self, amount: Decimal) -> Transaction:
        """Пополняет счёт на положительную сумму."""

        amount = normalize_money(amount)

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма пополнения должна быть положительной")

        self._balance += amount

        return self._add_history(
            operation_type=OperationType.DEPOSIT,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=OperationStatus.SUCCESS,
        )

    def withdraw(self, amount: Decimal) -> Transaction:
        """Снимает средства в пределах доступной суммы."""

        amount = normalize_money(amount)

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма снятия должна быть положительной")

        if self.is_blocked:
            status = OperationStatus.FAIL
            failure_reason = FailureReason.ACCOUNT_BLOCKED

        elif amount > self._get_available_withdrawal_funds():
            status = OperationStatus.FAIL
            failure_reason = self._get_insufficient_funds_reason()

        else:
            self._balance -= amount
            status = OperationStatus.SUCCESS
            failure_reason = None

        credit_used = self._get_credit_used(status)

        return self._add_history(
            operation_type=OperationType.WITHDRAW,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=status,
            failure_reason=failure_reason,
            credit_used=credit_used,
        )

    # Управление состоянием счёта

    def block_account(
        self,
        block_reason: BlockReason,
        actor: EventActor,
    ) -> AccountEvent | None:
        """Блокирует счёт по указанной причине."""

        if not isinstance(block_reason, BlockReason):
            raise TypeError("Причина блокировки должна иметь тип BlockReason")

        if not isinstance(actor, EventActor):
            raise TypeError("Инициатор должен иметь тип EventActor")

        if self.is_blocked:
            return None

        if not self._can_block(block_reason, actor):
            raise PermissionError(
                "Инициатор не имеет права блокировать счёт по этой причине"
            )

        self._account_status = AccountStatus.BLOCKED
        self._block_reason = block_reason

        return self._add_account_event(
            event_type=AccountEventType.BLOCKED,
            actor=actor,
            block_reason=block_reason,
        )

    def unblock_account(
        self,
        actor: EventActor,
    ) -> AccountEvent | None:
        """Разблокирует счёт."""

        if not isinstance(actor, EventActor):
            raise TypeError("Инициатор должен иметь тип EventActor")

        if not self.is_blocked:
            return None

        if not self._can_unblock(actor):
            raise PermissionError("Инициатор не имеет права разблокировать этот счёт")

        self._account_status = AccountStatus.ACTIVE
        self._block_reason = None

        return self._add_account_event(
            event_type=AccountEventType.UNBLOCKED,
            actor=actor,
        )

    # Получение истории

    def get_history(self) -> tuple[Transaction, ...]:
        """Возвращает неизменяемую историю операций."""

        return tuple(self._operations_history)

    def get_events_history(self) -> tuple[AccountEvent, ...]:
        """Возвращает неизменяемую историю событий."""

        return tuple(self._events_history)

    # Расширяемая логика снятия

    def _get_available_withdrawal_funds(self) -> Decimal:
        """Возвращает сумму, доступную для снятия."""

        return self._balance

    def _get_insufficient_funds_reason(
        self,
    ) -> FailureReason:
        """Возвращает причину отказа при нехватке средств."""

        return FailureReason.INSUFFICIENT_FUNDS

    def _get_credit_used(
        self,
        status: OperationStatus,
    ) -> bool | None:
        """Возвращает признак использования кредита."""

        return None

    # Проверка полномочий

    def _is_account_holder(
        self,
        actor: EventActor,
    ) -> bool:
        """Проверяет принадлежность счёта инициатору."""

        return (
            actor.actor_type is ActorType.ACCOUNT_HOLDER
            and actor.actor_id == self._holder_id
        )

    def _can_block(
        self,
        block_reason: BlockReason,
        actor: EventActor,
    ) -> bool:
        """Проверяет право инициатора заблокировать счёт."""

        if block_reason is BlockReason.USER_REQUEST:
            return (
                self._is_account_holder(actor)
                or actor.actor_type is ActorType.BANK_EMPLOYEE
            )

        allowed_actor_types = {
            BlockReason.FRAUD_SUSPICION: {
                ActorType.BANK_EMPLOYEE,
                ActorType.SYSTEM,
            },
            BlockReason.BANK_DECISION: {
                ActorType.BANK_EMPLOYEE,
            },
            BlockReason.LEGAL_RESTRICTION: {
                ActorType.BANK_EMPLOYEE,
                ActorType.SYSTEM,
            },
        }

        return actor.actor_type in allowed_actor_types[block_reason]

    def _can_unblock(
        self,
        actor: EventActor,
    ) -> bool:
        """Проверяет право инициатора разблокировать счёт."""

        if self._block_reason is BlockReason.USER_REQUEST:
            return (
                self._is_account_holder(actor)
                or actor.actor_type is ActorType.BANK_EMPLOYEE
            )

        return actor.actor_type is ActorType.BANK_EMPLOYEE

    # Внутренняя регистрация записей

    def _add_history(
        self,
        operation_type: OperationType,
        amount: Decimal,
        date_time: datetime,
        balance_after: Decimal,
        status: OperationStatus,
        failure_reason: FailureReason | None = None,
        credit_used: bool | None = None,
    ) -> Transaction:
        """Добавляет банковскую операцию в историю."""

        transaction = Transaction(
            transaction_id=uuid4(),
            account_id=self._account_id,
            currency=self._currency,
            operation_type=operation_type,
            amount=amount,
            created_at=date_time,
            balance_after=balance_after,
            status=status,
            failure_reason=failure_reason,
            credit_used=credit_used,
        )

        self._operations_history.append(transaction)

        return transaction

    def _add_account_event(
        self,
        event_type: AccountEventType,
        actor: EventActor,
        block_reason: BlockReason | None = None,
    ) -> AccountEvent:
        """Добавляет событие в историю счёта."""

        event = AccountEvent(
            event_id=uuid4(),
            account_id=self._account_id,
            event_type=event_type,
            created_at=datetime.now(timezone.utc),
            actor=actor,
            block_reason=block_reason,
        )

        self._events_history.append(event)

        return event


class CreditAccount(Account):
    """Моделирует банковский счёт с кредитным лимитом."""

    def __init__(
        self,
        account_holder: str,
        balance: Decimal = ZERO_MONEY,
        credit_limit: Decimal = ZERO_MONEY,
        *,
        currency: Currency = Currency.RUB,
        holder_id: UUID | None = None,
        account_id: UUID | None = None,
        account_status: AccountStatus = AccountStatus.ACTIVE,
        block_reason: BlockReason | None = None,
    ) -> None:
        """Создаёт банковский счёт с кредитным лимитом."""

        super().__init__(
            account_holder=account_holder,
            balance=balance,
            currency=currency,
            holder_id=holder_id,
            account_id=account_id,
            account_status=account_status,
            block_reason=block_reason,
        )

        credit_limit = normalize_money(credit_limit)

        if credit_limit < ZERO_MONEY:
            raise ValueError("Кредитный лимит не может быть отрицательным")

        self._credit_limit: Decimal = credit_limit

    def __str__(self) -> str:
        """Возвращает строковое представление счёта."""

        available_funds = self.get_available_funds()

        return (
            f"Владелец: {self.holder}. "
            f"Баланс: {self._balance} "
            f"{self._currency.value}. "
            f"Кредитный лимит: {self._credit_limit} "
            f"{self._currency.value}. "
            f"Доступно средств: {available_funds} "
            f"{self._currency.value}."
        )

    # Получение информации о кредитном счёте

    def get_credit_limit(self) -> Decimal:
        """Возвращает кредитный лимит."""

        return self._credit_limit

    def get_available_funds(self) -> Decimal:
        """Возвращает собственные и кредитные средства."""

        return self._balance + self._credit_limit

    # Переопределение логики снятия

    def _get_available_withdrawal_funds(self) -> Decimal:
        """Возвращает средства с кредитным лимитом."""

        return self.get_available_funds()

    def _get_insufficient_funds_reason(
        self,
    ) -> FailureReason:
        """Возвращает причину превышения лимита."""

        return FailureReason.CREDIT_LIMIT_EXCEEDED

    def _get_credit_used(
        self,
        status: OperationStatus,
    ) -> bool | None:
        """Показывает использование кредитных средств."""

        return status is OperationStatus.SUCCESS and self._balance < ZERO_MONEY

In [19]:
from dataclasses import FrozenInstanceError
from typing import Callable


def assert_raises(
    exception_type: type[BaseException],
    action: Callable[[], object],
) -> BaseException:
    """Проверяет, что вызов завершается указанным исключением."""

    try:
        action()
    except exception_type as error:
        return error
    except Exception as error:
        raise AssertionError(
            f"Ожидалось исключение {exception_type.__name__}, "
            f"получено {type(error).__name__}"
        ) from error

    raise AssertionError(f"Ожидалось исключение {exception_type.__name__}")


def test_normalize_money() -> None:
    """Проверяет нормализацию денежных значений."""

    assert normalize_money(Decimal("10.125")) == Decimal("10.12")

    assert normalize_money(Decimal("10.135")) == Decimal("10.14")

    assert_raises(
        TypeError,
        lambda: normalize_money(10.12),
    )

    assert_raises(
        ValueError,
        lambda: normalize_money(Decimal("Infinity")),
    )


def test_account_creation() -> None:
    """Проверяет создание обычного банковского счёта."""

    # Arrange
    holder_id = uuid4()
    account_id = uuid4()

    # Act
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.126"),
        account_id=account_id,
        holder_id=holder_id,
    )

    # Assert
    assert account.get_balance() == Decimal("100.13")
    assert account.get_account_id() == account_id
    assert account.get_holder_id() == holder_id
    assert account.get_currency() is Currency.RUB
    assert account.get_account_status() is AccountStatus.ACTIVE
    assert account.get_block_reason() is None
    assert account.get_history() == ()
    assert account.get_events_history() == ()


def test_account_constructor_validation() -> None:
    """Проверяет валидацию параметров банковского счёта."""

    assert_raises(
        ValueError,
        lambda: Account(
            account_holder="Илья",
            balance=Decimal("-1.00"),
        ),
    )

    assert_raises(
        TypeError,
        lambda: Account(
            account_holder="Илья",
            balance=Decimal("0.00"),
            holder_id="не UUID",  # type: ignore[arg-type]
        ),
    )

    assert_raises(
        ValueError,
        lambda: Account(
            account_holder="Илья",
            account_status=AccountStatus.BLOCKED,
        ),
    )

    assert_raises(
        ValueError,
        lambda: Account(
            account_holder="Илья",
            block_reason=BlockReason.BANK_DECISION,
        ),
    )


def test_deposit() -> None:
    """Проверяет успешное пополнение счёта."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Act
    transaction = account.deposit(Decimal("25.126"))

    # Assert
    assert transaction.operation_type is OperationType.DEPOSIT
    assert transaction.status is OperationStatus.SUCCESS
    assert transaction.amount == Decimal("25.13")
    assert transaction.balance_after == Decimal("125.13")
    assert transaction.account_id == account.get_account_id()
    assert transaction.currency is Currency.RUB
    assert transaction.failure_reason is None
    assert transaction.is_successful

    assert account.get_balance() == Decimal("125.13")
    assert account.get_history() == (transaction,)

    assert transaction.created_at.tzinfo is not None
    assert transaction.created_at.utcoffset() is not None


def test_invalid_deposit() -> None:
    """Проверяет запрет нулевого и отрицательного пополнения."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Act and Assert
    assert_raises(
        ValueError,
        lambda: account.deposit(Decimal("0.00")),
    )

    assert_raises(
        ValueError,
        lambda: account.deposit(Decimal("-1.00")),
    )

    assert account.get_balance() == Decimal("100.00")
    assert account.get_history() == ()


def test_successful_withdrawal() -> None:
    """Проверяет успешное снятие денег с обычного счёта."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Act
    transaction = account.withdraw(Decimal("40.00"))

    # Assert
    assert transaction.operation_type is OperationType.WITHDRAW
    assert transaction.status is OperationStatus.SUCCESS
    assert transaction.failure_reason is None
    assert transaction.credit_used is None
    assert transaction.balance_after == Decimal("60.00")

    assert account.get_balance() == Decimal("60.00")


def test_insufficient_funds() -> None:
    """Проверяет отказ при недостатке денег на обычном счёте."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Act
    transaction = account.withdraw(Decimal("101.00"))

    # Assert
    assert transaction.status is OperationStatus.FAIL

    assert transaction.failure_reason is FailureReason.INSUFFICIENT_FUNDS

    assert not transaction.is_successful
    assert transaction.balance_after == Decimal("100.00")
    assert account.get_balance() == Decimal("100.00")

    assert account.get_history() == (transaction,)


def test_credit_account() -> None:
    """Проверяет снятие денег с кредитного счёта."""

    # Arrange
    account = CreditAccount(
        account_holder="Илья",
        balance=Decimal("100.00"),
        credit_limit=Decimal("200.00"),
    )

    # Act
    successful_transaction = account.withdraw(Decimal("250.00"))

    # Assert
    assert successful_transaction.status is OperationStatus.SUCCESS
    assert successful_transaction.credit_used is True
    assert successful_transaction.failure_reason is None

    assert account.get_balance() == Decimal("-150.00")
    assert account.get_credit_limit() == Decimal("200.00")
    assert account.get_available_funds() == Decimal("50.00")

    # Act
    failed_transaction = account.withdraw(Decimal("50.01"))

    # Assert
    assert failed_transaction.status is OperationStatus.FAIL

    assert failed_transaction.failure_reason is FailureReason.CREDIT_LIMIT_EXCEEDED

    assert failed_transaction.credit_used is False
    assert account.get_balance() == Decimal("-150.00")


def test_event_actor_validation() -> None:
    """Проверяет создание инициаторов событий."""

    system = EventActor(
        actor_type=ActorType.SYSTEM,
    )

    assert system.actor_id is None

    assert_raises(
        ValueError,
        lambda: EventActor(
            actor_type=ActorType.SYSTEM,
            actor_id=uuid4(),
        ),
    )

    assert_raises(
        ValueError,
        lambda: EventActor(
            actor_type=ActorType.ACCOUNT_HOLDER,
        ),
    )


def test_holder_permissions() -> None:
    """Проверяет права владельца конкретного счёта."""

    # Arrange
    holder_id = uuid4()

    owner = EventActor(
        actor_type=ActorType.ACCOUNT_HOLDER,
        actor_id=holder_id,
    )

    stranger = EventActor(
        actor_type=ActorType.ACCOUNT_HOLDER,
        actor_id=uuid4(),
    )

    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
        holder_id=holder_id,
    )

    # Другой клиент не может заблокировать счёт
    assert_raises(
        PermissionError,
        lambda: account.block_account(
            block_reason=BlockReason.USER_REQUEST,
            actor=stranger,
        ),
    )

    assert not account.is_blocked
    assert account.get_block_reason() is None
    assert account.get_events_history() == ()

    # Настоящий владелец может заблокировать счёт
    blocked_event = account.block_account(
        block_reason=BlockReason.USER_REQUEST,
        actor=owner,
    )

    assert blocked_event is not None
    assert account.is_blocked

    assert account.get_block_reason() is BlockReason.USER_REQUEST

    assert blocked_event.actor == owner

    events_before = account.get_events_history()

    # Другой клиент не может разблокировать счёт
    assert_raises(
        PermissionError,
        lambda: account.unblock_account(stranger),
    )

    assert account.is_blocked
    assert account.get_events_history() == events_before

    # Настоящий владелец может разблокировать счёт
    unblocked_event = account.unblock_account(owner)

    assert unblocked_event is not None
    assert not account.is_blocked
    assert account.get_block_reason() is None

    assert account.get_events_history() == (
        blocked_event,
        unblocked_event,
    )


def test_employee_and_system_permissions() -> None:
    """Проверяет полномочия сотрудника и системы."""

    # Arrange
    employee = EventActor(
        actor_type=ActorType.BANK_EMPLOYEE,
        actor_id=uuid4(),
    )

    system = EventActor(
        actor_type=ActorType.SYSTEM,
    )

    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Система блокирует подозрительный счёт
    blocked_event = account.block_account(
        block_reason=BlockReason.FRAUD_SUSPICION,
        actor=system,
    )

    assert blocked_event is not None
    assert account.is_blocked

    # Система не может снять блокировку
    assert_raises(
        PermissionError,
        lambda: account.unblock_account(system),
    )

    assert account.is_blocked

    # Сотрудник снимает блокировку
    unblocked_event = account.unblock_account(employee)

    assert unblocked_event is not None
    assert not account.is_blocked
    assert account.get_block_reason() is None


def test_blocked_account_withdrawal() -> None:
    """Проверяет запрет снятия с заблокированного счёта."""

    # Arrange
    employee = EventActor(
        actor_type=ActorType.BANK_EMPLOYEE,
        actor_id=uuid4(),
    )

    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    account.block_account(
        block_reason=BlockReason.BANK_DECISION,
        actor=employee,
    )

    # Act
    transaction = account.withdraw(Decimal("10.00"))

    # Assert
    assert transaction.status is OperationStatus.FAIL

    assert transaction.failure_reason is FailureReason.ACCOUNT_BLOCKED

    assert transaction.balance_after == Decimal("100.00")
    assert account.get_balance() == Decimal("100.00")


def test_history_is_protected() -> None:
    """Проверяет защиту истории и банковской операции."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    transaction = account.deposit(Decimal("10.00"))

    # Act
    history = account.get_history()

    # Assert
    assert isinstance(history, tuple)
    assert history == (transaction,)

    assert_raises(
        AttributeError,
        lambda: history.append(transaction),  # type: ignore[attr-defined]
    )

    assert_raises(
        FrozenInstanceError,
        lambda: setattr(
            transaction,
            "amount",
            Decimal("999.00"),
        ),
    )


def test_blocked_account_accepts_deposit() -> None:
    """Проверяет пополнение заблокированного счёта."""

    # Arrange
    employee = EventActor(
        actor_type=ActorType.BANK_EMPLOYEE,
        actor_id=uuid4(),
    )

    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    account.block_account(
        block_reason=BlockReason.BANK_DECISION,
        actor=employee,
    )

    # Act
    transaction = account.deposit(Decimal("50.00"))

    # Assert
    assert transaction.status is OperationStatus.SUCCESS
    assert transaction.amount == Decimal("50.00")
    assert transaction.balance_after == Decimal("150.00")

    assert account.get_balance() == Decimal("150.00")
    assert account.is_blocked


def test_transaction_validation() -> None:
    """Проверяет валидацию банковской транзакции."""

    transaction_data = {
        "transaction_id": uuid4(),
        "account_id": uuid4(),
        "currency": Currency.RUB,
        "operation_type": OperationType.DEPOSIT,
        "amount": Decimal("10.00"),
        "created_at": datetime.now(timezone.utc),
        "balance_after": Decimal("10.00"),
        "status": OperationStatus.SUCCESS,
    }

    invalid_id_data = transaction_data | {
        "transaction_id": "не UUID",
    }

    assert_raises(
        TypeError,
        lambda: Transaction(**invalid_id_data),
    )

    invalid_amount_type = transaction_data | {
        "amount": 10.0,
    }

    assert_raises(
        TypeError,
        lambda: Transaction(**invalid_amount_type),
    )

    invalid_precision = transaction_data | {
        "amount": Decimal("10.001"),
    }

    assert_raises(
        ValueError,
        lambda: Transaction(**invalid_precision),
    )

    naive_datetime = transaction_data | {
        "created_at": datetime.now(),
    }

    assert_raises(
        ValueError,
        lambda: Transaction(**naive_datetime),
    )

    failed_without_reason = transaction_data | {
        "status": OperationStatus.FAIL,
    }

    assert_raises(
        ValueError,
        lambda: Transaction(**failed_without_reason),
    )

    successful_with_reason = transaction_data | {
        "failure_reason": FailureReason.ACCOUNT_BLOCKED,
    }

    assert_raises(
        ValueError,
        lambda: Transaction(**successful_with_reason),
    )


def test_account_event_validation() -> None:
    """Проверяет валидацию события счёта."""

    employee = EventActor(
        actor_type=ActorType.BANK_EMPLOYEE,
        actor_id=uuid4(),
    )

    event_data = {
        "event_id": uuid4(),
        "account_id": uuid4(),
        "event_type": AccountEventType.BLOCKED,
        "created_at": datetime.now(timezone.utc),
        "actor": employee,
        "block_reason": BlockReason.BANK_DECISION,
    }

    invalid_event_id = event_data | {
        "event_id": "не UUID",
    }

    assert_raises(
        TypeError,
        lambda: AccountEvent(**invalid_event_id),
    )

    invalid_actor = event_data | {
        "actor": "не EventActor",
    }

    assert_raises(
        TypeError,
        lambda: AccountEvent(**invalid_actor),
    )

    naive_datetime = event_data | {
        "created_at": datetime.now(),
    }

    assert_raises(
        ValueError,
        lambda: AccountEvent(**naive_datetime),
    )

    unblock_with_reason = event_data | {
        "event_type": AccountEventType.UNBLOCKED,
    }

    assert_raises(
        ValueError,
        lambda: AccountEvent(**unblock_with_reason),
    )

    block_without_reason = event_data | {
        "block_reason": None,
    }

    assert_raises(
        ValueError,
        lambda: AccountEvent(**block_without_reason),
    )


def test_block_permissions_matrix() -> None:
    """Проверяет все правила блокировки счёта."""

    permissions = (
        (
            BlockReason.USER_REQUEST,
            ActorType.ACCOUNT_HOLDER,
            True,
        ),
        (
            BlockReason.USER_REQUEST,
            ActorType.BANK_EMPLOYEE,
            True,
        ),
        (
            BlockReason.USER_REQUEST,
            ActorType.SYSTEM,
            False,
        ),
        (
            BlockReason.FRAUD_SUSPICION,
            ActorType.ACCOUNT_HOLDER,
            False,
        ),
        (
            BlockReason.FRAUD_SUSPICION,
            ActorType.BANK_EMPLOYEE,
            True,
        ),
        (
            BlockReason.FRAUD_SUSPICION,
            ActorType.SYSTEM,
            True,
        ),
        (
            BlockReason.BANK_DECISION,
            ActorType.ACCOUNT_HOLDER,
            False,
        ),
        (
            BlockReason.BANK_DECISION,
            ActorType.BANK_EMPLOYEE,
            True,
        ),
        (
            BlockReason.BANK_DECISION,
            ActorType.SYSTEM,
            False,
        ),
        (
            BlockReason.LEGAL_RESTRICTION,
            ActorType.ACCOUNT_HOLDER,
            False,
        ),
        (
            BlockReason.LEGAL_RESTRICTION,
            ActorType.BANK_EMPLOYEE,
            True,
        ),
        (
            BlockReason.LEGAL_RESTRICTION,
            ActorType.SYSTEM,
            True,
        ),
    )

    for reason, actor_type, is_allowed in permissions:
        holder_id = uuid4()

        account = Account(
            account_holder="Илья",
            balance=Decimal("100.00"),
            holder_id=holder_id,
        )

        if actor_type is ActorType.SYSTEM:
            actor = EventActor(
                actor_type=actor_type,
            )

        elif actor_type is ActorType.ACCOUNT_HOLDER:
            actor = EventActor(
                actor_type=actor_type,
                actor_id=holder_id,
            )

        else:
            actor = EventActor(
                actor_type=actor_type,
                actor_id=uuid4(),
            )

        if is_allowed:
            event = account.block_account(
                block_reason=reason,
                actor=actor,
            )

            assert event is not None
            assert account.is_blocked
            assert account.get_block_reason() is reason

        else:
            assert_raises(
                PermissionError,
                lambda reason=reason, actor=actor: account.block_account(
                    block_reason=reason,
                    actor=actor,
                ),
            )

            assert not account.is_blocked
            assert account.get_block_reason() is None
            assert account.get_events_history() == ()


def test_format_moscow_datetime() -> None:
    """Проверяет преобразование UTC во время Москвы."""

    utc_datetime = datetime(
        2026,
        1,
        10,
        12,
        30,
        tzinfo=timezone.utc,
    )

    result = format_moscow_datetime(utc_datetime)

    assert result == ("10.01.2026 15:30:00 MSK (UTC+03:00, Europe/Moscow)")


def test_format_moscow_datetime_rejects_naive_value() -> None:
    """Проверяет запрет времени без часового пояса."""

    naive_datetime = datetime(
        2026,
        1,
        10,
        12,
        30,
    )

    assert_raises(
        ValueError,
        lambda: format_moscow_datetime(naive_datetime),
    )

In [20]:
def run_all_tests() -> None:
    """Последовательно запускает все тесты программы."""

    tests = (
        test_normalize_money,
        test_account_creation,
        test_account_constructor_validation,
        test_deposit,
        test_invalid_deposit,
        test_successful_withdrawal,
        test_insufficient_funds,
        test_credit_account,
        test_event_actor_validation,
        test_holder_permissions,
        test_employee_and_system_permissions,
        test_blocked_account_withdrawal,
        test_history_is_protected,
        test_blocked_account_accepts_deposit,
        test_transaction_validation,
        test_account_event_validation,
        test_block_permissions_matrix,
        test_format_moscow_datetime,
        test_format_moscow_datetime_rejects_naive_value,
    )

    for test in tests:
        test()
        print(f"✅ {test.__name__}")

    print(f"\nВсе тесты пройдены: {len(tests)}")


run_all_tests()

✅ test_normalize_money
✅ test_account_creation
✅ test_account_constructor_validation
✅ test_deposit
✅ test_invalid_deposit
✅ test_successful_withdrawal
✅ test_insufficient_funds
✅ test_credit_account
✅ test_event_actor_validation
✅ test_holder_permissions
✅ test_employee_and_system_permissions
✅ test_blocked_account_withdrawal
✅ test_history_is_protected
✅ test_blocked_account_accepts_deposit
✅ test_transaction_validation
✅ test_account_event_validation
✅ test_block_permissions_matrix
✅ test_format_moscow_datetime
✅ test_format_moscow_datetime_rejects_naive_value

Все тесты пройдены: 19


In [21]:
from zoneinfo import ZoneInfo


MOSCOW_TIMEZONE = ZoneInfo("Europe/Moscow")


def format_moscow_datetime(value: datetime) -> str:
    """Форматирует время в часовом поясе Москвы."""

    if value.tzinfo is None or value.utcoffset() is None:
        raise ValueError("Дата и время должны содержать информацию о часовом поясе")

    moscow_datetime = value.astimezone(MOSCOW_TIMEZONE)
    offset = moscow_datetime.utcoffset()

    if offset is None:
        raise ValueError("Не удалось определить смещение часового пояса")

    total_minutes = int(offset.total_seconds() // 60)
    sign = "+" if total_minutes >= 0 else "-"
    absolute_minutes = abs(total_minutes)
    hours, minutes = divmod(absolute_minutes, 60)

    timezone_name = moscow_datetime.tzname() or "MSK"

    return (
        f"{moscow_datetime:%d.%m.%Y %H:%M:%S} "
        f"{timezone_name} "
        f"(UTC{sign}{hours:02d}:{minutes:02d}, "
        f"{MOSCOW_TIMEZONE.key})"
    )


def format_money(
    value: Decimal,
    currency: Currency,
) -> str:
    """Форматирует денежную сумму с указанием валюты."""

    formatted_value = f"{value:,.2f}".replace(",", " ")

    return f"{formatted_value} {currency.value}"


def format_transaction_history(account: Account) -> str:
    """Формирует текстовое представление истории транзакций."""

    history = account.get_history()

    header = (
        "ИСТОРИЯ ТРАНЗАКЦИЙ\n"
        f"Счёт: {account.get_account_id()}\n"
        f"Владелец: {account.holder}\n"
        f"Валюта: {account.get_currency().value}\n"
        f"{'=' * 72}"
    )

    if not history:
        return f"{header}\nИстория транзакций пуста."

    operation_names = {
        OperationType.DEPOSIT: "Пополнение",
        OperationType.WITHDRAW: "Снятие",
    }

    status_names = {
        OperationStatus.SUCCESS: "Успешно",
        OperationStatus.FAIL: "Отказ",
    }

    failure_reason_names = {
        FailureReason.INSUFFICIENT_FUNDS: ("Недостаточно средств"),
        FailureReason.CREDIT_LIMIT_EXCEEDED: ("Превышен кредитный лимит"),
        FailureReason.ACCOUNT_BLOCKED: ("Счёт заблокирован"),
    }

    lines = [header]

    for number, transaction in enumerate(
        history,
        start=1,
    ):
        operation_name = operation_names[transaction.operation_type]
        status_name = status_names[transaction.status]

        amount = format_money(
            transaction.amount,
            transaction.currency,
        )
        balance = format_money(
            transaction.balance_after,
            transaction.currency,
        )
        created_at = format_moscow_datetime(transaction.created_at)

        lines.extend(
            (
                f"\nТранзакция № {number}",
                f"ID: {transaction.transaction_id}",
                f"Операция: {operation_name}",
                f"Сумма: {amount}",
                f"Статус: {status_name}",
                f"Баланс после операции: {balance}",
                f"Время: {created_at}",
            )
        )

        if transaction.failure_reason is not None:
            failure_reason = failure_reason_names[transaction.failure_reason]
            lines.append(f"Причина отказа: {failure_reason}")

        if transaction.credit_used is not None:
            credit_used = "да" if transaction.credit_used else "нет"
            lines.append(f"Использованы кредитные средства: {credit_used}")

        lines.append("-" * 72)

    return "\n".join(lines)


def format_account_events(account: Account) -> str:
    """Формирует текстовое представление событий счёта."""

    events = account.get_events_history()

    header = (
        "ИСТОРИЯ СОБЫТИЙ СЧЁТА\n"
        f"Счёт: {account.get_account_id()}\n"
        f"Владелец: {account.holder}\n"
        f"{'=' * 72}"
    )

    if not events:
        return f"{header}\nИстория событий пуста."

    event_names = {
        AccountEventType.BLOCKED: ("Счёт заблокирован"),
        AccountEventType.UNBLOCKED: ("Счёт разблокирован"),
    }

    actor_names = {
        ActorType.ACCOUNT_HOLDER: ("Владелец счёта"),
        ActorType.BANK_EMPLOYEE: ("Сотрудник банка"),
        ActorType.SYSTEM: ("Банковская система"),
    }

    block_reason_names = {
        BlockReason.USER_REQUEST: ("Запрос владельца"),
        BlockReason.FRAUD_SUSPICION: ("Подозрение на мошенничество"),
        BlockReason.BANK_DECISION: ("Решение банка"),
        BlockReason.LEGAL_RESTRICTION: ("Юридическое ограничение"),
    }

    lines = [header]

    for number, event in enumerate(
        events,
        start=1,
    ):
        event_name = event_names[event.event_type]
        actor_name = actor_names[event.actor.actor_type]
        created_at = format_moscow_datetime(event.created_at)

        lines.extend(
            (
                f"\nСобытие № {number}",
                f"ID: {event.event_id}",
                f"Событие: {event_name}",
                f"Инициатор: {actor_name}",
                f"Время: {created_at}",
            )
        )

        if event.actor.actor_id is not None:
            lines.append(f"ID инициатора: {event.actor.actor_id}")

        if event.block_reason is not None:
            block_reason = block_reason_names[event.block_reason]
            lines.append(f"Причина блокировки: {block_reason}")

        lines.append("-" * 72)

    return "\n".join(lines)


def format_account_journal(account: Account) -> str:
    """Формирует полный журнал транзакций и событий счёта."""

    return f"{format_transaction_history(account)}\n\n{format_account_events(account)}"

In [22]:
def demonstrate_banking_program() -> None:
    """Демонстрирует основные возможности банковской программы."""

    print("=" * 72)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ БАНКОВСКОЙ ПРОГРАММЫ")
    print("=" * 72)

    # Создание участников
    holder_id = uuid4()

    owner = EventActor(
        actor_type=ActorType.ACCOUNT_HOLDER,
        actor_id=holder_id,
    )

    employee = EventActor(
        actor_type=ActorType.BANK_EMPLOYEE,
        actor_id=uuid4(),
    )

    system = EventActor(
        actor_type=ActorType.SYSTEM,
    )

    # Создание кредитного счёта
    account = CreditAccount(
        account_holder="Илья",
        balance=Decimal("1000.00"),
        credit_limit=Decimal("500.00"),
        currency=Currency.RUB,
        holder_id=holder_id,
    )

    print("\n1. Создан кредитный счёт")
    print(account)
    print(f"ID счёта: {account.get_account_id()}")
    print(f"ID владельца: {account.get_holder_id()}")
    print(f"Статус счёта: {account.get_account_status().value}")

    # Пополнение
    deposit_transaction = account.deposit(Decimal("250.50"))

    print("\n2. Выполнено пополнение")
    print(f"ID транзакции: {deposit_transaction.transaction_id}")
    print(
        "Сумма: "
        f"{
            format_money(
                deposit_transaction.amount,
                deposit_transaction.currency,
            )
        }"
    )
    print(f"Статус: {deposit_transaction.status.value}")
    print(
        "Баланс: "
        f"{
            format_money(
                account.get_balance(),
                account.get_currency(),
            )
        }"
    )
    print(f"Время: {format_moscow_datetime(deposit_transaction.created_at)}")

    # Успешное снятие
    withdrawal_transaction = account.withdraw(Decimal("400.00"))

    print("\n3. Выполнено успешное снятие")
    print(f"ID транзакции: {withdrawal_transaction.transaction_id}")
    print(
        "Сумма: "
        f"{
            format_money(
                withdrawal_transaction.amount,
                withdrawal_transaction.currency,
            )
        }"
    )
    print(f"Статус: {withdrawal_transaction.status.value}")
    print(
        "Баланс: "
        f"{
            format_money(
                account.get_balance(),
                account.get_currency(),
            )
        }"
    )
    print(f"Время: {format_moscow_datetime(withdrawal_transaction.created_at)}")

    # Снятие с использованием кредитных средств
    credit_transaction = account.withdraw(Decimal("1000.00"))

    print("\n4. Выполнено снятие кредитных средств")
    print(f"ID транзакции: {credit_transaction.transaction_id}")
    print(
        "Сумма: "
        f"{
            format_money(
                credit_transaction.amount,
                credit_transaction.currency,
            )
        }"
    )
    print(f"Статус: {credit_transaction.status.value}")
    print(
        "Использованы кредитные средства: "
        f"{'да' if credit_transaction.credit_used else 'нет'}"
    )
    print(
        "Баланс: "
        f"{
            format_money(
                account.get_balance(),
                account.get_currency(),
            )
        }"
    )
    print(
        "Доступно средств: "
        f"{
            format_money(
                account.get_available_funds(),
                account.get_currency(),
            )
        }"
    )

    # Неуспешное снятие из-за превышения лимита
    failed_credit_transaction = account.withdraw(Decimal("400.00"))

    print("\n5. Попытка превысить кредитный лимит")
    print(f"Статус: {failed_credit_transaction.status.value}")

    if failed_credit_transaction.failure_reason is not None:
        print(f"Причина отказа: {failed_credit_transaction.failure_reason.value}")

    print(
        "Баланс не изменился: "
        f"{
            format_money(
                account.get_balance(),
                account.get_currency(),
            )
        }"
    )

    # Пополнение для восстановления положительного баланса
    repayment_transaction = account.deposit(Decimal("500.00"))

    print("\n6. Выполнено погашение задолженности")
    print(
        "Сумма: "
        f"{
            format_money(
                repayment_transaction.amount,
                repayment_transaction.currency,
            )
        }"
    )
    print(
        "Баланс: "
        f"{
            format_money(
                account.get_balance(),
                account.get_currency(),
            )
        }"
    )

    # Блокировка владельцем
    blocked_event = account.block_account(
        block_reason=BlockReason.USER_REQUEST,
        actor=owner,
    )

    print("\n7. Счёт заблокирован по просьбе владельца")
    print(f"Статус счёта: {account.get_account_status().value}")

    block_reason = account.get_block_reason()

    if block_reason is not None:
        print(f"Причина: {block_reason.value}")

    if blocked_event is not None:
        print(f"ID события: {blocked_event.event_id}")
        print(f"Время: {format_moscow_datetime(blocked_event.created_at)}")

    # Попытка снятия с заблокированного счёта
    blocked_withdrawal = account.withdraw(Decimal("50.00"))

    print("\n8. Попытка снятия с заблокированного счёта")
    print(f"Статус: {blocked_withdrawal.status.value}")

    if blocked_withdrawal.failure_reason is not None:
        print(f"Причина отказа: {blocked_withdrawal.failure_reason.value}")

    print(
        "Баланс не изменился: "
        f"{
            format_money(
                account.get_balance(),
                account.get_currency(),
            )
        }"
    )

    # Владелец разблокирует счёт
    unblocked_event = account.unblock_account(owner)

    print("\n9. Счёт разблокирован владельцем")
    print(f"Статус счёта: {account.get_account_status().value}")
    print(f"Причина блокировки: {account.get_block_reason()}")

    if unblocked_event is not None:
        print(f"ID события: {unblocked_event.event_id}")
        print(f"Время: {format_moscow_datetime(unblocked_event.created_at)}")

    # Системная блокировка
    system_blocked_event = account.block_account(
        block_reason=BlockReason.FRAUD_SUSPICION,
        actor=system,
    )

    print("\n10. Счёт заблокирован системой")
    print(f"Статус счёта: {account.get_account_status().value}")

    if system_blocked_event is not None:
        print(f"Инициатор: {system_blocked_event.actor.actor_type.value}")
        print(f"Причина: {system_blocked_event.block_reason.value}")

    # Попытка системы снять установленную блокировку
    print("\n11. Система пытается разблокировать счёт")

    try:
        account.unblock_account(system)
    except PermissionError as error:
        print(f"Отказ в доступе: {error}")

    print(f"Счёт остаётся заблокированным: {account.is_blocked}")

    # Сотрудник банка снимает системную блокировку
    employee_unblocked_event = account.unblock_account(employee)

    print("\n12. Сотрудник банка разблокировал счёт")
    print(f"Статус счёта: {account.get_account_status().value}")

    if employee_unblocked_event is not None:
        print(f"Инициатор: {employee_unblocked_event.actor.actor_type.value}")
        print(f"Время: {format_moscow_datetime(employee_unblocked_event.created_at)}")

    # Итоговая информация
    print("\n" + "=" * 72)
    print("ИТОГОВОЕ СОСТОЯНИЕ СЧЁТА")
    print("=" * 72)
    print(account)
    print(f"Количество транзакций: {len(account.get_history())}")
    print(f"Количество событий: {len(account.get_events_history())}")

    # Полный журнал
    print("\n")
    print(format_account_journal(account))


demonstrate_banking_program()

ДЕМОНСТРАЦИЯ РАБОТЫ БАНКОВСКОЙ ПРОГРАММЫ

1. Создан кредитный счёт
Владелец: Илья. Баланс: 1000.00 RUB. Кредитный лимит: 500.00 RUB. Доступно средств: 1500.00 RUB.
ID счёта: db08adc9-d47f-4171-9135-54c02d4ce161
ID владельца: 3d5723fb-5516-4d32-8ed5-ebad13f9865c
Статус счёта: active

2. Выполнено пополнение
ID транзакции: e81a3c96-9dab-490c-878e-97b90b9d3201
Сумма: 250.50 RUB
Статус: success
Баланс: 1 250.50 RUB
Время: 10.10.2026 18:02:50 MSK (UTC+03:00, Europe/Moscow)

3. Выполнено успешное снятие
ID транзакции: 34182101-23f9-4777-a562-6d609117cd01
Сумма: 400.00 RUB
Статус: success
Баланс: 850.50 RUB
Время: 10.10.2026 18:02:50 MSK (UTC+03:00, Europe/Moscow)

4. Выполнено снятие кредитных средств
ID транзакции: 003489a7-753d-4d36-a517-95e8258dbc6c
Сумма: 1 000.00 RUB
Статус: success
Использованы кредитные средства: да
Баланс: -149.50 RUB
Доступно средств: 350.50 RUB

5. Попытка превысить кредитный лимит
Статус: fail
Причина отказа: credit_limit_exceeded
Баланс не изменился: -149.50 RUB
